In [ ]:
# =====================
# CNN-GNN (LOCAL JUPYTER)
# =====================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

from torch_geometric.nn import GCNConv

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

# =====================
# CONFIG
# =====================
EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-3
K = 5
DATA_DIR = '../data/kakao'   # ⬅️ relatif dari folder notebooks
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Device:", DEVICE)


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
NUM_CLASSES = len(dataset.classes)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))

idx = np.arange(len(dataset))
train_idx, temp_idx = train_test_split(
    idx, test_size=0.3, stratify=dataset.targets, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=2/3,
    stratify=np.array(dataset.targets)[temp_idx], random_state=42
)

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)


class CNN_GNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = SimpleCNN()
        self.gcn1 = GCNConv(128, 256)
        self.gcn2 = GCNConv(256, num_classes)

    def forward(self, images, edge_index):
        x = self.cnn(images)
        x = F.relu(self.gcn1(x, edge_index))
        return self.gcn2(x, edge_index)


In [ ]:
def build_knn_graph(features, k=5):
    sim = cosine_similarity(features)
    edges = []

    for i in range(len(features)):
        neighbors = np.argsort(sim[i])[-(k+1):]
        for j in neighbors:
            if i != j:
                edges.append([i, j])
                edges.append([j, i])

    return torch.tensor(edges, dtype=torch.long).t().contiguous()


In [ ]:
model = CNN_GNN(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

train_acc, val_acc = [], []

for epoch in range(EPOCHS):
    # ---- TRAIN ----
    model.train()
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        with torch.no_grad():
            feats = model.cnn(images).cpu().numpy()
            edge_index = build_knn_graph(feats, K).to(DEVICE)

        out = model(images, edge_index)
        loss = criterion(out, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc.append(correct / total)

    # ---- VALIDATION ----
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            feats = model.cnn(images).cpu().numpy()
            edge_index = build_knn_graph(feats, K).to(DEVICE)

            out = model(images, edge_index)
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)

    val_acc.append(correct / total)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Acc: {train_acc[-1]:.4f} | "
          f"Val Acc: {val_acc[-1]:.4f}")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(train_acc, label='Train')
plt.plot(val_acc, label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('CNN-GNN Performance')
plt.legend()
plt.grid(True)
plt.show()
